# Численное решение системы нелинейных уравнений методом Ньютона

В данном разделе рассматривается решение системы двух нелинейных уравнений вида:
$$
\begin{cases} 
f_1(x, y) = 0 \\
f_2(x, y) = 0 
\end{cases}
$$

### 1. Постановка задачи
Мы ищем вектор состояния $\mathbf{u} = (x, y)^T$, который обращает вектор-функцию $\mathbf{F}(\mathbf{u}) = (f_1, f_2)^T$ в ноль. В коде используются функции:
* $f_1(x, y) = x^5 + y^4 - 2$
* $f_2(x, y) = (x - 2)^3 + (y - 2)^3 + 16$

### 2. Метод Ньютона (линеаризация)
Для нахождения следующего приближения $\mathbf{u}_{k+1} = \mathbf{u}_k + \Delta \mathbf{u}$ используется разложение в ряд Тейлора. Решается система линейных уравнений относительно поправок $(\Delta x, \Delta y)$:
$$
J(\mathbf{u}_k) \Delta \mathbf{u} = -\mathbf{F}(\mathbf{u}_k)
$$
где $J$ — матрица Якоби:
$$
J = \begin{pmatrix} 
\frac{\partial f_1}{\partial x} & \frac{\partial f_1}{\partial y} \\ 
\frac{\partial f_2}{\partial x} & \frac{\partial f_2}{\partial y} 
\end{pmatrix}
$$

Поправки вычисляются по правилу Крамера (как реализовано в функции `newton_step`):
$$
\Delta x = \frac{\det J_x}{\det J}, \quad \Delta y = \frac{\det J_y}{\det J}
$$

### 3. Модификация: Метод Федоренко
Для улучшения сходимости и предотвращения расходимости при плохом начальном приближении используется параметр релаксации (шаг) $\tau \in (0, 1]$:
$$
\mathbf{u}_{k+1} = \mathbf{u}_k + \tau \Delta \mathbf{u}
$$

В реализации применен алгоритм адаптивного выбора шага: если значение нормы не уменьшается, шаг $\tau$ дробится ($\tau = \tau / 2$) до выполнения условия:
$$
\|\mathbf{F}(\mathbf{u}_{k+1})\| < \|\mathbf{F}(\mathbf{u}_{k})\|
$$

### 4. Критерий сходимости
В качестве меры близости к решению используется взвешенная норма:
$$
\|\mathbf{F}\|_\alpha = \sqrt{\left(\frac{f_1}{\alpha_1}\right)^2 + \left(\frac{f_2}{\alpha_2}\right)^2}
$$
где веса $\alpha_i$ — это евклидовы нормы градиентов соответствующих функций $\alpha_i = \|\nabla f_i\|_2$. Это позволяет отнормировать вклад каждого уравнения в общую ошибку.

Итерационный процесс прекращается, когда $\|\mathbf{F}\|_\alpha < \varepsilon$.

In [1]:
import numpy as np 
import matplotlib.pyplot as plt 
from tqdm.notebook import tqdm
import plotly.graph_objects as go

def f1(x, y): 
    return x**5 + y**4 - 2 
 
def f2(x, y): 
    return (x - 2)**3 + (y - 2)**3 + 16 
 
def df1_dx(x, y): 
    return 5 * x**4 
 
def df1_dy(x, y): 
    return 4 * y**3 
 
def df2_dx(x, y): 
    return 3 * (x - 2)**2 
 
def df2_dy(x, y): 
    return 3 * (y - 2)**2 
 
def weighted_norm(x, y): 
    a1 = np.sqrt(df1_dx(x, y)**2 + df1_dy(x, y)**2) 
    a2 = np.sqrt(df2_dx(x, y)**2 + df2_dy(x, y)**2) 
    return np.sqrt((f1(x, y)/a1)**2 + (f2(x, y)/a2)**2) 
 
def newton_step(x, y): 
    a = df1_dx(x, y) 
    b = df1_dy(x, y) 
    c = df2_dx(x, y) 
    d = df2_dy(x, y) 
 
    F1 = f1(x, y) 
    F2 = f2(x, y) 
 
    det = a * d - b * c 
 
    dx = ((-F1) * d - b * (-F2)) / det 
    dy = (a * (-F2) - (-F1) * c) / det 
 
    return dx, dy 
 


def newton(x,y,eps=1e-15, max_iter=100,tau=1.0):
    norms = [weighted_norm(x, y)] 
    print("-"*50)
    print(f'tau = {tau}')
    for k in range(max_iter): 
        dx, dy = newton_step(x, y) 
        norm_old = weighted_norm(x, y) 
    
        while True: 
            x_new = x + tau * dx 
            y_new = y + tau * dy 
            if weighted_norm(x_new, y_new) < norm_old: 
                break 
            tau /= 2 
    
        x, y = x_new, y_new 
        norms.append(weighted_norm(x, y)) 
        print(f'Иттерация = {k}, ошибка = {norms[-1]}')
    
        if norms[-1] < eps: 
            break 
    print()
    print()
    return norms

x, y = 2.0, 3.0 
# tau_arr = np.arange(0.5,2,0.1)
tau_arr = [1]
results = []
for tau in tqdm(tau_arr):
    results.append(newton(x,y,tau=tau))


 



fig = go.Figure()

for tau, norms in zip(tau_arr, results):
    fig.add_trace(go.Scatter(
        y=norms,
        x=list(range(len(norms))),
        mode='lines+markers',
        name=f"τ = {tau}"
    ))

fig.update_layout(
    title="Сходимость метода Ньютона–Федоренко",
    xaxis_title="Номер итерации k",
    yaxis_title="Взвешенная норма |F(x_k)|_α",
    # yaxis_type="log", 
)

fig.show()

  0%|          | 0/1 [00:00<?, ?it/s]

--------------------------------------------------
tau = 1
Иттерация = 0, ошибка = 2.037328534838945
Иттерация = 1, ошибка = 1.4245092961480592
Иттерация = 2, ошибка = 1.1021622463648086
Иттерация = 3, ошибка = 0.8787952335837056
Иттерация = 4, ошибка = 0.6999285135386544
Иттерация = 5, ошибка = 0.5424059511452943
Иттерация = 6, ошибка = 0.4188944685786476
Иттерация = 7, ошибка = 0.3115212457250974
Иттерация = 8, ошибка = 0.20164533568198315
Иттерация = 9, ошибка = 0.08869918333449647
Иттерация = 10, ошибка = 0.01562857436772656
Иттерация = 11, ошибка = 0.00043247435794686856
Иттерация = 12, ошибка = 3.2142897162187434e-07
Иттерация = 13, ошибка = 1.7712158095459493e-13
Иттерация = 14, ошибка = 2.6053355513225265e-17


